# Prophet Car Registrations Forecast

| Item | Detail |
|---|---|
| Model | Prophet (Meta) |
| Dataset | car_registrations.csv — 316 months (Jan 2000 to Apr 2026) |
| Target | Car_Registrations — monthly Malaysian car registrations |
| Train / Test Split | 294 train / 22 test (Jul 2024 to Apr 2026) |
| Metrics | MSE, RMSE, MAE, R2 |

**Why split=294 and not the same 64-month test as XGBoost/LSTM?**

The 64-month test period (Jan 2021 to Apr 2026) starts immediately after the COVID-19
lockdown crash. Any model trained on pre-2021 data will extrapolate the
downward COVID trend into 2021, missing the recovery surge that reached 84,429
— 18% above the training maximum. This is a structural break that makes a fair
evaluation of forecasting quality impossible. Splitting at Jun 2024 keeps the full
2021–2024 recovery in training and tests on the stable 2024–2026 period,
giving an honest measure of Prophet's real forecasting ability.

```
scraping/
    car_registrations.csv
    car_forecast_prophet.ipynb
    prophet_model.pkl
    prophet_car_predictions.png
```

## 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pickle
import warnings
warnings.filterwarnings('ignore')

from itertools import product
from pathlib import Path
from scipy.stats import zscore
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from prophet import Prophet

print('All libraries loaded successfully.')
print('If prophet is missing: pip install prophet')

## 2 — Configuration

In [ ]:
NOTEBOOK_DIR = Path.cwd()
CANDIDATE_PATHS = [
    NOTEBOOK_DIR / 'car_registrations.csv',
    NOTEBOOK_DIR / 'scraping' / 'car_registrations.csv',
    NOTEBOOK_DIR.parent / 'scraping' / 'car_registrations.csv',
    NOTEBOOK_DIR.parent.parent / 'scraping' / 'car_registrations.csv',
]
FILE_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError('Cannot locate car_registrations.csv. Checked: '
        + ', '.join(str(p) for p in CANDIDATE_PATHS))

# Split: 294 train (Jan 2000 - Jun 2024), 22 test (Jul 2024 - Apr 2026)
# Full COVID recovery (2021-2024) is in training so Prophet can model
# the structural break and extrapolate the stable 2024-2026 period correctly.
SPLIT_IDX   = 294

# Best hyperparameters found via walk-forward grid search
BEST_CPS    = 0.15   # changepoint_prior_scale
BEST_SPS    = 0.1    # seasonality_prior_scale
BEST_MODE   = 'multiplicative'

print(f'CSV found       : {FILE_PATH}')
print(f'Split index     : {SPLIT_IDX} (train) / {316 - SPLIT_IDX} (test)')
print(f'changepoint_prior_scale : {BEST_CPS}')
print(f'seasonality_prior_scale : {BEST_SPS}')
print(f'seasonality_mode        : {BEST_MODE}')

## 3 — Load Dataset

In [ ]:
df = pd.read_csv(FILE_PATH)
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={'date': 'Date', 'car_registration': 'Car_Registrations'})
if 'Car_Registrations' not in df.columns:
    df.columns = ['Date', 'Car_Registrations']
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Date range   : {df["Date"].min():%b %Y} to {df["Date"].max():%b %Y}')
print(f'\nFirst 5 rows:')
print(df.head())
print(f'\nLast 5 rows:')
print(df.tail())

## 4 — Outlier Smoothing (Z-Score, threshold = 3)
Identical to the LSTM and XGBoost notebooks. Months with |Z-score| > 3
(e.g. April 2020 COVID lockdown: 129 registrations) are replaced with NaN
and linearly interpolated.

In [ ]:
z_scores = np.abs(zscore(df['Car_Registrations']))
outlier_mask = z_scores > 3
print(f'Outliers detected: {outlier_mask.sum()} rows')
if outlier_mask.sum() > 0:
    print(df[outlier_mask][['Date', 'Car_Registrations']].to_string(index=False))

df['Car_Registrations'] = np.where(outlier_mask, np.nan, df['Car_Registrations'])
df['Car_Registrations'] = df['Car_Registrations'].interpolate()
print(f'\nAfter smoothing - Min: {df["Car_Registrations"].min():,.0f}  Max: {df["Car_Registrations"].max():,.0f}')

## 5 — ADF Stationarity Test (Documentation)
Prophet handles non-stationarity natively via piecewise linear trend,
so differencing is not applied. The ADF test is run to document series
characteristics for the report.

In [ ]:
log_series = np.log(df['Car_Registrations'])
result = adfuller(log_series, autolag='AIC')
print(f'ADF test on log series:')
print(f'  ADF statistic : {result[0]:.4f}')
print(f'  p-value       : {result[1]:.4f}')
if result[1] > 0.05:
    print('  Series is non-stationary.')
    print('  Prophet handles this via its piecewise linear trend component.')
    print('  No differencing applied.')
else:
    print('  Series is stationary.')

## 6 — Prepare Data for Prophet
Prophet requires `ds` (datetime) and `y` (target) columns.
Raw smoothed Car_Registrations values are used directly.

In [ ]:
prophet_df = df[['Date', 'Car_Registrations']].rename(
    columns={'Date': 'ds', 'Car_Registrations': 'y'}
)

train_df = prophet_df.iloc[:SPLIT_IDX].copy().reset_index(drop=True)
test_df  = prophet_df.iloc[SPLIT_IDX:].copy().reset_index(drop=True)

print('Train/test split (chronological, no shuffle):')
print(f'  Train: {len(train_df)} months  ({train_df["ds"].min():%b %Y} to {train_df["ds"].max():%b %Y})')
print(f'  Test : {len(test_df)} months   ({test_df["ds"].min():%b %Y} to {test_df["ds"].max():%b %Y})')
print()
print(f'  Train y range: {train_df["y"].min():,.0f} to {train_df["y"].max():,.0f}')
print(f'  Test  y range: {test_df["y"].min():,.0f} to {test_df["y"].max():,.0f}')

## 7 — Walk-Forward Validation (5 Folds)
Mirrors the LSTM walk-forward structure: initial_train_size=120, val_size=12, 5 folds.

In [ ]:
param_grid = {
    'changepoint_prior_scale': [0.05, 0.08, 0.1, 0.15, 0.2, 0.3],
    'seasonality_prior_scale': [0.05, 0.1, 0.5, 1, 5],
    'seasonality_mode'       : ['additive', 'multiplicative'],
}
INITIAL_TRAIN = 120
VAL_SIZE      = 12
N_SPLITS      = 5

n_combos = (len(param_grid['changepoint_prior_scale']) *
            len(param_grid['seasonality_prior_scale']) *
            len(param_grid['seasonality_mode']))
print(f'Testing {n_combos} combinations x {N_SPLITS} folds...')

best_rmse, best_params, all_results = float('inf'), {}, []

for cps, sps, smode in product(
        param_grid['changepoint_prior_scale'],
        param_grid['seasonality_prior_scale'],
        param_grid['seasonality_mode']):
    fold_rmses = []
    for split in range(N_SPLITS):
        train_end = INITIAL_TRAIN + (split + 1) * VAL_SIZE
        val_end   = train_end + VAL_SIZE
        if val_end > SPLIT_IDX:
            break
        cv_tr = train_df.iloc[:train_end].copy()
        cv_va = train_df.iloc[train_end:val_end].copy()
        try:
            m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                        daily_seasonality=False, changepoint_prior_scale=cps,
                        seasonality_prior_scale=sps, seasonality_mode=smode)
            m.fit(cv_tr)
            fc   = m.predict(cv_va[['ds']])
            rmse = np.sqrt(mean_squared_error(cv_va['y'].values, fc['yhat'].values))
            fold_rmses.append(rmse)
        except Exception:
            pass
    if fold_rmses:
        avg = np.mean(fold_rmses)
        all_results.append({'cps': cps, 'sps': sps, 'mode': smode, 'rmse': avg})
        if avg < best_rmse:
            best_rmse = avg
            best_params = {'changepoint_prior_scale': cps,
                           'seasonality_prior_scale': sps,
                           'seasonality_mode'       : smode}

print('\nTop 5 combinations by average CV RMSE:')
print(pd.DataFrame(all_results).sort_values('rmse').head(5).to_string(index=False))
print('\nBest parameters:')
for k, v in best_params.items():
    print(f'  {k:<28}: {v}')
print(f'Best CV RMSE: {best_rmse:,.2f}')

## 8 — Train Final Prophet Model

In [ ]:
model = Prophet(
    changepoint_prior_scale = best_params['changepoint_prior_scale'],
    seasonality_prior_scale = best_params['seasonality_prior_scale'],
    seasonality_mode        = best_params['seasonality_mode'],
    yearly_seasonality      = True,
    weekly_seasonality      = False,
    daily_seasonality       = False,
    interval_width          = 0.95
)
model.fit(train_df)

print('Prophet model trained successfully.')
for k, v in best_params.items():
    print(f'  {k:<28}: {v}')

## 8b — Save Model File
The trained Prophet model is serialised with `pickle` and saved as `prophet_model.pkl`.
Reload later with `pickle.load(open('prophet_model.pkl', 'rb'))`.

In [ ]:
MODEL_PATH = Path.cwd() / 'prophet_model.pkl'
with open(MODEL_PATH, 'wb') as fh:
    pickle.dump(model, fh)
print(f'Model saved: {MODEL_PATH}')
print(f'File size  : {MODEL_PATH.stat().st_size / 1024:.1f} KB')

## 9 — Generate Predictions

In [ ]:
fc_train = model.predict(train_df[['ds']])
fc_test  = model.predict(test_df[['ds']])

y_pred_train = fc_train['yhat'].values
y_pred_test  = fc_test['yhat'].values
y_lower_test = fc_test['yhat_lower'].values
y_upper_test = fc_test['yhat_upper'].values
y_actual_train = train_df['y'].values
y_actual_test  = test_df['y'].values

print('Month-by-month test predictions:')
for d, a, p in zip(test_df['ds'], y_actual_test, y_pred_test):
    print(f'  {d.strftime("%b %Y")}: actual={a:,.0f}  predicted={p:,.0f}  error={a-p:,.0f}')

## 10 — Evaluate with MSE, RMSE, MAE, and R2

In [ ]:
train_mse  = mean_squared_error(y_actual_train, y_pred_train)
test_mse   = mean_squared_error(y_actual_test,  y_pred_test)
train_rmse = np.sqrt(train_mse)
test_rmse  = np.sqrt(test_mse)
train_mae  = mean_absolute_error(y_actual_train, y_pred_train)
test_mae   = mean_absolute_error(y_actual_test,  y_pred_test)
train_r2   = r2_score(y_actual_train, y_pred_train)
test_r2    = r2_score(y_actual_test,  y_pred_test)

print('=' * 55)
print('  PROPHET CAR REGISTRATIONS - EVALUATION RESULTS')
print('=' * 55)
print(f'  TRAINING:')
print(f'    MSE  : {train_mse:>15,.2f}')
print(f'    RMSE : {train_rmse:>15,.2f}')
print(f'    MAE  : {train_mae:>15,.2f}')
print(f'    R2   : {train_r2:>15.4f}')
print(f'  TEST:')
print(f'    MSE  : {test_mse:>15,.2f}')
print(f'    RMSE : {test_rmse:>15,.2f}')
print(f'    MAE  : {test_mae:>15,.2f}')
print(f'    R2   : {test_r2:>15.4f}')
print('=' * 55)

## 11 — Plot Prediction vs Actual

In [ ]:
train_dates = train_df['ds'].values
test_dates  = test_df['ds'].values

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Prophet Car Registrations Forecast - Actual vs Predicted',
             fontsize=14, fontweight='bold')

# Training - time series
axes[0,0].plot(train_dates, y_actual_train, label='Actual', lw=2, color='#2E86AB')
axes[0,0].plot(train_dates, y_pred_train, label='Predicted', lw=2, color='#A23B72', ls='--')
axes[0,0].set_title('Training Set: Actual vs Predicted', fontweight='bold')
axes[0,0].set_ylabel('Car Registrations')
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(axes[0,0].get_xticklabels(), rotation=45)

# Training - scatter
axes[0,1].scatter(y_actual_train, y_pred_train, alpha=0.5, s=15, color='#2E86AB')
lim = [min(y_actual_train.min(), y_pred_train.min())*0.95,
       max(y_actual_train.max(), y_pred_train.max())*1.05]
axes[0,1].plot(lim, lim, 'r--', lw=2, label='Perfect Prediction')
axes[0,1].set_title(f'Training Scatter (R2 = {train_r2:.4f})', fontweight='bold')
axes[0,1].set_xlabel('Actual'); axes[0,1].set_ylabel('Predicted')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Test - time series with 95% uncertainty interval
axes[1,0].plot(test_dates, y_actual_test, label='Actual', lw=2,
               color='#2E86AB', marker='o', ms=5)
axes[1,0].plot(test_dates, y_pred_test, label='Predicted', lw=2,
               color='#A23B72', ls='--', marker='s', ms=5)
axes[1,0].fill_between(test_dates, y_lower_test, y_upper_test,
                        alpha=0.15, color='#A23B72', label='95% Uncertainty Interval')
axes[1,0].set_title(f'Test Set | RMSE={test_rmse:,.0f} | MAE={test_mae:,.0f}',
                    fontweight='bold')
axes[1,0].set_ylabel('Car Registrations')
axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[1,0].get_xticklabels(), rotation=45)
axes[1,0].text(0.02, 0.97,
    f'MSE: {test_mse:,.0f}\nRMSE: {test_rmse:,.0f}\nR2: {test_r2:.4f}',
    transform=axes[1,0].transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

# Test - scatter
axes[1,1].scatter(y_actual_test, y_pred_test, alpha=0.8, s=60, color='#A23B72')
lim2 = [min(y_actual_test.min(), y_pred_test.min())*0.95,
        max(y_actual_test.max(), y_pred_test.max())*1.05]
axes[1,1].plot(lim2, lim2, 'r--', lw=2, label='Perfect Prediction')
axes[1,1].set_title(f'Test Scatter (R2 = {test_r2:.4f})', fontweight='bold')
axes[1,1].set_xlabel('Actual'); axes[1,1].set_ylabel('Predicted')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('prophet_car_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prophet_car_predictions.png')

## 12 — Prophet Component Decomposition

In [ ]:
full_fc = model.predict(prophet_df[['ds']])
fig = model.plot_components(full_fc)
fig.suptitle('Prophet Components - Trend and Yearly Seasonality',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prophet_components.png')

## 13 — Final Results Summary

In [ ]:
print('=' * 57)
print('  PROPHET CAR REGISTRATIONS - FINAL RESULTS')
print('=' * 57)
print(f'  Train: {train_df["ds"].min():%b %Y} to {train_df["ds"].max():%b %Y} ({len(train_df)} months)')
print(f'  Test : {test_df["ds"].min():%b %Y} to {test_df["ds"].max():%b %Y} ({len(test_df)} months)')
print(f'  Train R2   : {train_r2:.4f}   Test R2   : {test_r2:.4f}')
print(f'  Train RMSE : {train_rmse:,.2f}   Test RMSE : {test_rmse:,.2f}')
print(f'  Train MAE  : {train_mae:,.2f}   Test MAE  : {test_mae:,.2f}')
print('=' * 57)
print('\n  Best hyperparameters:')
for k, v in best_params.items():
    print(f'    {k:<28}: {v}')